# BNN on Pynq

This notebook covers how to use Binary Neural Networks on Pynq. 
It shows an example of image recognition with a Binarized Neural Network inspired at VGG-16, featuring 6 convolutional layers, 3 max pool layers and 3 fully connected layers

## 1. Instantiate a Classifier
Creating a classifier will automatically download the correct bitstream onto device and load the weights trained on the specified dataset. By default there are three sets of weights available for the CNV network using 1 bit for weights and activation - this example uses the streetview house number set.

In [ ]:
import bnn
print(bnn.available_params(bnn.NETWORK_CNVW1A1))

classifier = bnn.CnvClassifier(bnn.NETWORK_CNVW1A1,"streetview",bnn.RUNTIME_HW)

## 2. Get classes of dataset

In [ ]:
print(classifier.classes)

## 3. Open image to be classified

The image to be classified is loaded and shown:

In [ ]:
from PIL import Image
# Compatibility with the original BNN-PYNQ image formatter.
if not hasattr(Image, 'ANTIALIAS'):
    Image.ANTIALIAS = Image.Resampling.LANCZOS
import numpy as np
img = Image.open('/home/xilinx/jupyter_notebooks/bnn/pictures/6.png')
img

## 4. Launching BNN in hardware
The image is passed to PL and the inference is performed. The python API will automatically take care of formatting the picture to the right size that can be processed by CNV network.

In [ ]:
result_class_idx = classifier.classify_image(img)
print("Inferred number: {0}".format(classifier.class_name(result_class_idx)))

## 5. Launching BNN in software
The inference on the same image is performed in sofware on the ARM core:

In [ ]:
sw_classifier = bnn.CnvClassifier(bnn.NETWORK_CNVW1A1, "streetview", bnn.RUNTIME_SW)
result_class_idx = sw_classifier.classify_image(img)
print("Inferred number: {0}".format(sw_classifier.class_name(result_class_idx)))

## 6. Test more real digits (0-9)

Run the next cell once to load **20 small, real SVHN test images** and define the display function. The collection is about **50 KB**, downloaded from this repository only on first use. There is no full dataset download and no training.

Then run the following cell to classify **one image of each digit, 0-9**, on the FPGA. Each image shows its dataset label and the network prediction: **green / OK** is correct, **red / wrong** is an error. The source image number is a zero-based index in the original SVHN test set.

The classifier recognises the labelled central digit even when neighbouring digits are visible. Its class index is not the digit itself: `class_name` maps the model's class order (1,...,9,0) to the digit.

In [ ]:
import base64
import contextlib
import hashlib
import io
import json
from pathlib import Path
import urllib.request
import matplotlib.pyplot as plt
from PIL import ImageFilter, ImageEnhance

# A small collection of 20 real SVHN images; cached after the first download.
sample_file = Path("/home/xilinx/jupyter_notebooks/bnn/pictures/svhn_examples.json")
sample_sha256 = "9394e68e00347dc47044a521742bb19e31db821ef9e5a863f9a78fb95f80fec8"
if not sample_file.exists() or hashlib.sha256(sample_file.read_bytes()).hexdigest() != sample_sha256:
    url = "https://raw.githubusercontent.com/onigas/BNN-PYNQ/master/notebooks/pictures/svhn_examples.json"
    print("Downloading 20 small SVHN examples...", flush=True)
    with urllib.request.urlopen(url, timeout=60) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != sample_sha256:
        raise ValueError("Sample file checksum mismatch.")
    sample_file.parent.mkdir(parents=True, exist_ok=True)
    sample_file.write_bytes(payload)
else:
    print("Using the existing SVHN examples; no download.")

examples = json.loads(sample_file.read_text())["samples"]
for example in examples:
    example["image"] = Image.open(io.BytesIO(base64.b64decode(example["png"]))).convert("RGB")

resampling = getattr(Image, "Resampling", Image)

def test_grid(cases, title, columns=5):
    columns = min(columns, len(cases))
    rows = (len(cases) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(2.4 * columns, 2.9 * rows),
                             squeeze=False)
    outcomes = []
    for ax in axes.flat:
        ax.axis("off")
    for ax, case in zip(axes.flat, cases):
        # Suppress repeated timing messages; keep the original image intact.
        with contextlib.redirect_stdout(io.StringIO()):
            index = classifier.classify_image(case["image"].copy())
        # Class index and digit differ: the model orders classes 1,...,9,0.
        prediction = int(classifier.class_name(int(index)))
        expected = int(case["digit"])
        correct = prediction == expected
        ax.imshow(case["image"], interpolation="nearest")
        ax.set_title(case["caption"], fontsize=9)
        ax.text(0.5, -0.12, "Label: {} | FPGA: {}".format(expected, prediction),
                transform=ax.transAxes, ha="center", fontsize=10,
                color="green" if correct else "red")
        ax.text(0.5, -0.26, "OK" if correct else "wrong",
                transform=ax.transAxes, ha="center", fontsize=9,
                color="green" if correct else "red")
        outcomes.append({"label": expected, "prediction": prediction,
                         "correct": correct, "caption": case["caption"]})
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=(0, 0.04, 1, 0.95), h_pad=3.5)
    plt.show()
    print("Correct in this selection: {} / {}".format(
        sum(row["correct"] for row in outcomes), len(outcomes)))
    return outcomes

print("Ready. Run the next cell to classify the first 10 images.")


In [ ]:
clear_cases = [
    {"image": e["image"], "digit": e["digit"],
     "caption": "SVHN test #{}".format(e["index"])}
    for e in examples if e["group"] == "clear"
]
clear_results = test_grid(clear_cases, "FPGA: different real digits, 0-9")

## 7. Real images that are harder to read

Run the next cell. These are **different original photographs**, not edited versions of section 6. They include blur, poor lighting, low contrast, slanted digits, and surrounding digits.

Compare the ground-truth label with the FPGA prediction. Some labels can be hard for a person to read too; they are taken from the dataset rather than guessed from the displayed thumbnail. This small, deliberately selected collection is not a measurement of full-dataset accuracy.

In [ ]:
difficult_cases = [
    {"image": e["image"], "digit": e["digit"],
     "caption": "#{}: {}".format(e["index"], e["note"])}
    for e in examples if e["group"] == "difficult"
]
difficult_results = test_grid(difficult_cases, "FPGA: difficult real SVHN images")

## 8. What happens when a digit is rotated?

Run the next cell. It takes the same three real images (digits **2, 5, 8**) and applies **-25°, -12°, 0°, +12°, +25°** rotations.

These are **artificial transformations**, separate from the natural variation above. The image is padded before rotation to reduce clipping and resized to the network's 32 × 32 input. The 0° reference receives the same padding and resizing. No model is retrained.

Compare predictions across angles. Errors are possible: the pretrained model is not guaranteed to be rotation invariant. We avoid upside-down 6/9 examples because their intended digit can become ambiguous.

In [ ]:
def rotated_input(image, angle):
    # Padding reduces clipping and is applied to the 0-degree reference too.
    pixels = np.asarray(image)
    border = np.concatenate((pixels[0], pixels[-1], pixels[:, 0], pixels[:, -1]))
    background = tuple(int(x) for x in np.median(border, axis=0))
    padded = Image.new("RGB", (48, 48), background)
    padded.paste(image, (8, 8))
    return padded.rotate(angle, resample=resampling.BICUBIC,
                         expand=False, fillcolor=background).resize(
                             (32, 32), resampling.LANCZOS)

rotation_cases = []
for digit in (2, 5, 8):
    original = next(e["image"] for e in examples
                    if e["group"] == "clear" and e["digit"] == digit)
    for angle in (-25, -12, 0, 12, 25):
        rotation_cases.append({
            "image": rotated_input(original, angle), "digit": digit,
            "caption": "Digit {}: {:+d} degrees".format(digit, angle)})
rotation_results = test_grid(rotation_cases, "FPGA: controlled rotation (synthetic)")

## 9. What happens with blur, low contrast, and darkness?

Run the next cell. For the same digits **2, 5, 8**, compare the original image with a blurred, low-contrast, and darker version. Only one effect is applied at a time.

These are controlled artificial changes. A changed prediction shows sensitivity to that condition; an unchanged prediction for three samples does not prove robustness on the full dataset.

In [ ]:
degraded_cases = []
for digit in (2, 5, 8):
    original = next(e["image"] for e in examples
                    if e["group"] == "clear" and e["digit"] == digit)
    variants = [
        ("original", original.copy()),
        ("blur", original.filter(ImageFilter.GaussianBlur(radius=1.2))),
        ("low contrast", ImageEnhance.Contrast(original).enhance(0.45)),
        ("darker", ImageEnhance.Brightness(original).enhance(0.45)),
    ]
    for name, image in variants:
        degraded_cases.append({"image": image, "digit": digit,
                               "caption": "Digit {}: {}".format(digit, name)})
degraded_results = test_grid(degraded_cases, "FPGA: blur and lighting (synthetic)", columns=4)

## 10. Short questions

1. Which difficult real images did the network classify incorrectly? Could you read their digits without the labels?
2. For each rotated digit, at which angles did its prediction change?
3. Which effect caused the most errors in this selection: blur, lower contrast, or darkness?

All new predictions above use **the FPGA classifier** loaded in section 1. Section 5 remains the original software comparison on the first image.

### Image sources

The 20 original images are lossless 32 × 32 PNGs selected from **SVHN, cropped digits, official test split**. The original dataset labels and row indices are included with the images. SVHN uses label 10 for digit 0; the supplied examples map it to 0.

- Dataset: http://ufldl.stanford.edu/housenumbers/
- Reference: Yuval Netzer et al., *Reading Digits in Natural Images with Unsupervised Feature Learning*, NIPS Workshop on Deep Learning and Unsupervised Feature Learning, 2011.
- Download mirror used for preparation: https://huggingface.co/datasets/darcook/DCPV-Artifacts/blob/main/data/svhn/test_32x32.mat
- Original test archive MD5 checked against the Torchvision dataset specification: `eb5a983be6a315427106f1b164d9cef3`.
- SVHN images retain the original dataset's non-commercial research and educational-use terms; the repository's code license does not relicense the images.
- Sections 8-9 generate labelled artificial variants locally.

**Validation:** sample provenance, labels, PNGs, transformations, and display code have been checked off-board. Actual FPGA predictions must be checked on the PYNQ board.